# Bước 07 — Ablation Roboflow Transmission Detection (YOLO11n@640)

Dataset mới: [Roboflow `school-vc1kl/transmission-detection` v1](https://universe.roboflow.com/school-vc1kl/transmission-detection) — CC BY 4.0, 1393 ảnh (đã bị Roboflow **stretch về 640×640**).

| Roboflow class | → class của mình |
|---|---|
| `cable_defectueux` (dây hỏng / xổ sợi) | `broken_strand` (5) |
| `rubbish` (diều, bạt, rác vướng dây) | `foreign_object` (4) |

**So sánh có kiểm soát:** TRAIN = đúng dataset của run 640 trước (`v6p5_compose_4b9a452e24e2` = V6.2E + 500 synthetic) **+ Roboflow train/valid**. Hyper-parameter, seed giống hệt run `20261008T181605Z_6b0eac5f` → chênh lệch chỉ do data Roboflow.

**Đánh giá trên 2 tập:**
1. V6.2E VAL (frozen, 2272 ảnh) — gate như bước 05.
2. Roboflow TEST (external_eval) — đã loại ảnh gần trùng với TRAIN; đo model có tổng quát sang domain dây thật không.

**Chống rò rỉ:** mọi ảnh Roboflow trùng SHA hoặc pHash ≤ 4 với **bất kỳ** ảnh V6.2E (train/val/test) đều bị loại.

Chạy trong kernel **WSL**, chạy lần lượt từ trên xuống. Cell TRAIN tự **resume** từ `last.pt` nếu bị ngắt giữa chừng (giữ nguyên `RUN_ID`).

In [ ]:
# 0. CONFIG
from pathlib import Path
import os, json, shutil, hashlib, collections
from datetime import datetime, timezone

PROJECT  = Path("/mnt/c/Users/intern/Documents/SEP490/train")
RF_SRC   = Path("/mnt/c/Users/intern/Downloads/transmission-detection.v1i.yolov11")   # bản export YOLO từ Roboflow
WORK     = Path.home() / "yolov11sdi_data"                     # ext4: nhanh + ổn định hơn /mnt/c
RF_DIR   = WORK / "external/roboflow_transmission"
BASE_DS  = Path.home() / "datasets/v6p5_compose_4b9a452e24e2"  # dataset của run 640 trước (đã có sẵn)
OUT_DS   = Path.home() / "datasets/v6p5_roboflow_ablation"
EVAL_DS  = Path.home() / "datasets/roboflow_external_eval"
INTERNAL_RAW = [PROJECT / "data/cache/base_extract", PROJECT / "data/cache/overlay_extract"]  # chứa cả V6.2E TEST
HASH_CACHE = WORK / "cache/v6p2e_internal_hashes.npz"          # tính 1 lần (~vài phút), lần sau dùng lại

NAMES = ["insulator", "insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand"]
RF_MAP = {"cable_defectueux": 5, "rubbish": 4}
PHASH_MAX = 4

# Giữ y hệt run 640 trước để so sánh công bằng
MODEL, IMGSZ, EPOCHS, BATCH, DEVICE, WORKERS, SEED = PROJECT / "legacy/notebooks/v6p5/yolo11n.pt", 640, 50, 8, 0, 2, 20261007
PREV_BEST = PROJECT / "artifacts/v6p5_train_local/20261008T181605Z_6b0eac5f/runs/yolo11n_v6p5_data_ablation_640/weights/best.pt"
BASELINE = {"mAP50": 0.768, "mAP50_95": 0.413,
            "foreign_object": {"AP50": 0.537, "AP50_95": 0.295}, "broken_strand": {"AP50": 0.749, "AP50_95": 0.356}}

# Đặt RUN_ID cũ vào đây để resume, None = run mới
RUN_ID = None
RUN_ID = RUN_ID or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_roboflow"
ROOT = PROJECT / "artifacts/v6p5_train_local" / RUN_ID
RUN_NAME = "yolo11n_v6p5_roboflow_ablation_640"
ROOT.mkdir(parents=True, exist_ok=True)
for p in (RF_SRC, BASE_DS / "data.yaml", MODEL, PREV_BEST):
    assert p.exists(), f"Thiếu: {p}"
print("RUN_ID:", RUN_ID, "\nOutput:", ROOT)

In [ ]:
# 1. COPY ROBOFLOW -> ext4 + ĐỌC LABEL (polygon -> bbox) + MAP CLASS
import yaml, numpy as np
from PIL import Image

if not (RF_DIR / "data.yaml").exists():
    shutil.copytree(RF_SRC, RF_DIR, dirs_exist_ok=True)
rf_cfg = yaml.safe_load((RF_DIR / "data.yaml").read_text())
rf_names = rf_cfg["names"] if isinstance(rf_cfg["names"], list) else [rf_cfg["names"][k] for k in sorted(rf_cfg["names"])]
cid_map = {i: RF_MAP[n] for i, n in enumerate(rf_names)}
print("Roboflow:", rf_cfg.get("roboflow", {}).get("license"), "| map:", {rf_names[i]: NAMES[c] for i, c in cid_map.items()})

def to_box(parts):
    v = [float(x) for x in parts[1:]]
    if len(v) == 4:
        xc, yc, w, h = v
    else:  # polygon x1 y1 x2 y2 ... -> bounding box
        xs, ys = v[0::2], v[1::2]
        x0, x1, y0, y1 = max(0, min(xs)), min(1, max(xs)), max(0, min(ys)), min(1, max(ys))
        xc, yc, w, h = (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0
    return xc, yc, w, h

rf_items = []   # (split, image path, [rows])
stats = collections.Counter()
for split in ("train", "valid", "test"):
    for ip in sorted((RF_DIR / split / "images").iterdir()):
        lp = RF_DIR / split / "labels" / (ip.stem + ".txt")
        rows = []
        for ln in (lp.read_text().splitlines() if lp.exists() else []):
            p = ln.split()
            if len(p) < 5:
                continue
            xc, yc, w, h = to_box(p)
            if w * 640 < 2 or h * 640 < 2:      # bỏ box suy biến
                stats["degenerate_dropped"] += 1
                continue
            rows.append((cid_map[int(p[0])], xc, yc, w, h))
            stats[f"{split}/{NAMES[cid_map[int(p[0])]]}"] += 1
            stats["polygon" if len(p) > 5 else "bbox"] += 1
        rf_items.append((split, ip, rows))
print(len(rf_items), "ảnh"); print(dict(sorted(stats.items())))

In [ ]:
# 2. DEDUP: Roboflow vs TOÀN BỘ V6.2E (exact SHA + pHash), và Roboflow TEST vs Roboflow TRAIN
import imagehash
from concurrent.futures import ThreadPoolExecutor
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def feat(p):
    try:
        with Image.open(p) as im:
            h = imagehash.phash(im.convert("RGB")).hash.flatten()
        return hashlib.sha256(p.read_bytes()).hexdigest(), np.packbits(h)
    except Exception:
        return None, None

if HASH_CACHE.exists():
    z = np.load(HASH_CACHE, allow_pickle=True)
    in_sha, in_ph, in_where = set(z["sha"].tolist()), z["ph"], z["where"]
    print("Dùng lại hash cache:", HASH_CACHE, len(in_ph))
else:
    internal = [p for p in (BASE_DS / "images").rglob("*") if p.suffix.lower() in IMG_EXTS]
    for r in INTERNAL_RAW:
        internal += [p for p in r.rglob("*") if p.suffix.lower() in IMG_EXTS]
    print("Hash", len(internal), "ảnh nội bộ (lần đầu mất vài phút vì đọc /mnt/c)...")
    with ThreadPoolExecutor(16) as ex:
        F = list(ex.map(feat, internal))
    keep = [i for i, f in enumerate(F) if f[0] is not None]
    sha_arr = np.array([F[i][0] for i in keep]); in_ph = np.stack([F[i][1] for i in keep])
    in_where = np.array([internal[i].as_posix() for i in keep])
    HASH_CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.savez(HASH_CACHE, sha=sha_arr, ph=in_ph, where=in_where)
    in_sha = set(sha_arr.tolist())

with ThreadPoolExecutor(16) as ex:
    rf_feat = list(ex.map(feat, [ip for _, ip, _ in rf_items]))
in_bits = np.unpackbits(in_ph, axis=1)

def min_dist(bits, pool):
    return (pool != bits).sum(1)

drop, audit = set(), []
for (split, ip, rows), (s, ph) in zip(rf_items, rf_feat):
    bits = np.unpackbits(ph)
    d = min_dist(bits, in_bits); j = int(d.argmin())
    reason = "exact_sha" if s in in_sha else ("phash" if d[j] <= PHASH_MAX else None)
    if reason:
        drop.add(ip)
    audit.append({"split": split, "image": ip.name, "min_phash_v6p2e": int(d[j]), "nearest": str(in_where[j]), "dropped": reason})

# Roboflow random-split các frame gần nhau -> TEST gần trùng TRAIN thì bỏ khỏi external eval
tr_bits = np.stack([np.unpackbits(f[1]) for (sp, ip, _), f in zip(rf_items, rf_feat) if sp != "test" and ip not in drop])
for a, (split, ip, rows), (s, ph) in zip(audit, rf_items, rf_feat):
    if split == "test" and ip not in drop:
        dt = int(min_dist(np.unpackbits(ph), tr_bits).min())
        a["min_phash_rf_train"] = dt
        if dt <= PHASH_MAX:
            drop.add(ip); a["dropped"] = "near_rf_train"

import pandas as pd
audit_df = pd.DataFrame(audit)
audit_df.to_csv(ROOT / "roboflow_dedup_audit.csv", index=False)
print(audit_df.groupby(["split"])["dropped"].value_counts(dropna=False).unstack(fill_value=0))
print("Khoảng cách pHash gần nhất tới V6.2E (0 = trùng):")
print(audit_df["min_phash_v6p2e"].clip(upper=12).value_counts().sort_index().to_dict())

In [ ]:
# 3. BUILD DATASET (hard link trên ext4 -> không tốn thêm dung lượng)
def link(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        os.link(src, dst)

def write_label(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("".join(f"{c} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n" for c, xc, yc, w, h in rows))

shutil.rmtree(OUT_DS, ignore_errors=True); shutil.rmtree(EVAL_DS, ignore_errors=True)
for kind in ("images", "labels"):
    for split in ("train", "val"):
        for p in (BASE_DS / kind / split).iterdir():
            link(p, OUT_DS / kind / split / p.name)

added = collections.Counter(); box_added = collections.Counter()
for split, ip, rows in rf_items:
    if ip in drop or not rows:
        continue
    name = "rf_" + hashlib.sha256(ip.read_bytes()).hexdigest()[:16]
    if split == "test":
        link(ip, EVAL_DS / "images/val" / (name + ip.suffix.lower()))
        write_label(EVAL_DS / "labels/val" / (name + ".txt"), rows); added["external_eval"] += 1
    else:
        link(ip, OUT_DS / "images/train" / (name + ip.suffix.lower()))
        write_label(OUT_DS / "labels/train" / (name + ".txt"), rows); added["train"] += 1
        for r in rows: box_added[NAMES[r[0]]] += 1

def write_yaml(d, train_rel):
    y = d / "data.yaml"
    y.write_text(yaml.safe_dump({"path": str(d), "train": train_rel, "val": "images/val",
                                 "names": dict(enumerate(NAMES))}, sort_keys=False))
    return y
data_yaml = write_yaml(OUT_DS, "images/train")
eval_yaml = write_yaml(EVAL_DS, "images/val")

n_train = len(list((OUT_DS / "images/train").iterdir())); n_val = len(list((OUT_DS / "images/val").iterdir()))
assert n_val == 2272, n_val
build = {"run_id": RUN_ID, "rf_added": dict(added), "rf_train_boxes": dict(box_added), "train_images": n_train,
         "val_images": n_val, "rf_dropped": len(drop), "base_dataset": str(BASE_DS)}
(ROOT / "build.json").write_text(json.dumps(build, indent=2))
print(json.dumps(build, indent=2))

In [ ]:
# 4. TRAIN — tự resume nếu đã có last.pt của RUN_ID này
from ultralytics import YOLO
last = ROOT / "runs" / RUN_NAME / "weights/last.pt"
if last.exists():
    print("RESUME:", last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO(str(MODEL)).train(
        data=str(data_yaml), imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, device=DEVICE, workers=WORKERS,
        cache=False, save=True, save_period=5, pretrained=True, seed=SEED,
        project=str(ROOT / "runs"), name=RUN_NAME, exist_ok=True, plots=False, verbose=True)
best = ROOT / "runs" / RUN_NAME / "weights/best.pt"
assert best.exists(), best
print("Best:", best)

In [ ]:
# 5. EVAL: model mới vs model run trước, trên V6.2E VAL và Roboflow external_eval
import gc, torch
gc.collect(); torch.cuda.empty_cache()
rows = []
for tag, w in (("roboflow_ablation", best), ("previous_v6p5", PREV_BEST)):
    for ds, y in (("v6p2e_val", data_yaml), ("rf_external_eval", eval_yaml)):
        m = YOLO(str(w)).val(data=str(y), split="val", imgsz=IMGSZ, batch=BATCH, device=DEVICE, plots=False,
                             verbose=False, project=str(ROOT / "eval"), name=f"{tag}_{ds}", exist_ok=True).box
        rows.append({"model": tag, "eval": ds, "class": "ALL", "AP50": m.map50, "AP50_95": m.map})
        for c, a50, a in zip(m.ap_class_index, m.ap50, m.ap):
            rows.append({"model": tag, "eval": ds, "class": NAMES[int(c)], "AP50": float(a50), "AP50_95": float(a)})
res = pd.DataFrame(rows)
res.to_csv(ROOT / "roboflow_ablation_eval.csv", index=False)
show = res[res["class"].isin(["ALL", "foreign_object", "broken_strand"])]
display(show.pivot_table(index=["eval", "class"], columns="model", values=["AP50", "AP50_95"]).round(3))

In [ ]:
# 6. GATE (giống bước 05, trên V6.2E VAL)
v = res[(res["model"] == "roboflow_ablation") & (res["eval"] == "v6p2e_val")].set_index("class")
gate = {
    "global_mAP50_95_not_drop_1pt": v.loc["ALL", "AP50_95"] >= BASELINE["mAP50_95"] - 0.01,
    "foreign_AP50_+2pt":   v.loc["foreign_object", "AP50"]    >= BASELINE["foreign_object"]["AP50"] + 0.02,
    "foreign_AP50_95_+1pt": v.loc["foreign_object", "AP50_95"] >= BASELINE["foreign_object"]["AP50_95"] + 0.01,
    "strand_AP50_95_+1pt": v.loc["broken_strand", "AP50_95"]  >= BASELINE["broken_strand"]["AP50_95"] + 0.01,
}
gate = {k: bool(x) for k, x in gate.items()}
(ROOT / "gate.json").write_text(json.dumps({"gate": gate, "pass": all(gate.values()), **build}, indent=2))
for k, x in gate.items(): print("✅" if x else "❌", k)
print("PASS" if all(gate.values()) else "FAIL", "| kết quả lưu ở", ROOT)

## Đọc kết quả
- **V6.2E VAL** là tiêu chí chính (gate). **Roboflow external_eval** cho biết model có nhận được dây hỏng/rác ở domain mới không — model cũ thường thấp ở đây; nếu model mới tăng mạnh ở external mà VAL không giảm → data có ích cho triển khai thật.
- Ảnh Roboflow bị **stretch 640×640** (méo tỉ lệ) và có nhiều frame liên tiếp từ cùng video (`cable3-###`) → số ảnh "độc lập" thực tế ít hơn 1393.
- Nếu foreign tăng mà strand giảm (hoặc ngược lại) → tách ablation từng class như ghi chú ở bước 05.